In [2]:
import os
from dotenv import load_dotenv


In [3]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated,Literal
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage, BaseMessage
from pydantic import BaseModel, Field
from langgraph.checkpoint.memory import MemorySaver
import operator

In [4]:
load_dotenv()

True

In [5]:
model = ChatGoogleGenerativeAI(model = "gemini-3.6-flash")

In [6]:
class jokeState(TypedDict):
    topic:str
    joke:str
    explanation:str
    

In [7]:
def generate_joke(state:jokeState):
    prompt = f'generate a joke on the following topic \n - {state["topic"]}'
    response = model.invoke(prompt).content
    return {'joke':response}


In [8]:
def generate_explanation(state:jokeState):
    prompt = f'generate an explanation on the joke {state["joke"]}'
    response = model.invoke(prompt).content
    return {'explanation':response}

In [9]:
graph = StateGraph(jokeState)

graph.add_node('generate_joke',generate_joke)
graph.add_node('generate_explanation',generate_explanation)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explanation')
graph.add_edge('generate_explanation',END)

checkpointer = MemorySaver()    

workflow = graph.compile(checkpointer=checkpointer )

In [10]:
config1 = {'configurable':{'thread_id':'1'}}
workflow.invoke({'topic':'pizza'},config = config1)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


KeyboardInterrupt: 

In [ ]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': "I was going to tell you a joke about pizza...\n\n...but it's all in the delivery!", 'extras': {'signature': 'EpceCpQeAWkUfRMfvTsFiICmQSo9QnR73myOfTT1aez+EjZ/WfaA5Z0tqmUU0puZVU/KbG5JxXmZXe6HnLMQcnhXiYGfT5kNlAmNZuXsrrDIlagRtbXK4nzUsWXaK72Ns2OrCsEv415KtVu1os/ByEVKPXuzWhdCX70KWMBeGswrh/49/cwcXRIaqLWsrbiyec4rcRRbDmx38wW+0/Ct49dAZmVNV1uzr2PDM22+C7UcPphlnh2W8UMACndUEOyIlTCOBE3ZVS8BeHje9nV9gHh9ittWiWo3qPl0EhRDvIKuA33u3/FxEGBVAzbFA9Sfg4zKZBgVqW/zSm7IvP5V9dNv7oYnRTqyAIxhUq5MK5mnuucq5AXd0Z+1LoxM8a87GJuN1gncZQdkAVMSmMaePumCN/vqkzKAd4ISBMMANrQNcy+B/yVlB/RzWLFjpHksrIKmyYxa36zRa0N7giqJdtufD+1Z5kBKdM3gI4ZQzOqX33ibz3zRSIi7MaKgk8hn8lLd1UC+k1dAXMyWjgGiiBOtoBboDLA9h8z9jzQtB6S0T3/0/lMnfFNhMGCAegES1SQBqSr6sAcz+rnRG5zQe20lghiJ9D2FURDr9tLXMzjp8+bahxoccIH35IFKWLciBt2Rt9c+BpWlF60hMxf+qeGyTgGwGlotvboyc3CWD67B9Gn8m3Y9SgsbpIw+tbNZ3OJEIyuRXzpDO0IaIBrdow9329sWkWy6RdXgW8AJFSDLerMo7O/ai9Rl8xNIZixO4yR/rDz1zs0Z13hEtutTeo9Bz/z9j7IJVlBArmRkW1LzGgdF

In [ ]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': "I was going to tell you a joke about pizza...\n\n...but it's all in the delivery!", 'extras': {'signature': 'EpceCpQeAWkUfRMfvTsFiICmQSo9QnR73myOfTT1aez+EjZ/WfaA5Z0tqmUU0puZVU/KbG5JxXmZXe6HnLMQcnhXiYGfT5kNlAmNZuXsrrDIlagRtbXK4nzUsWXaK72Ns2OrCsEv415KtVu1os/ByEVKPXuzWhdCX70KWMBeGswrh/49/cwcXRIaqLWsrbiyec4rcRRbDmx38wW+0/Ct49dAZmVNV1uzr2PDM22+C7UcPphlnh2W8UMACndUEOyIlTCOBE3ZVS8BeHje9nV9gHh9ittWiWo3qPl0EhRDvIKuA33u3/FxEGBVAzbFA9Sfg4zKZBgVqW/zSm7IvP5V9dNv7oYnRTqyAIxhUq5MK5mnuucq5AXd0Z+1LoxM8a87GJuN1gncZQdkAVMSmMaePumCN/vqkzKAd4ISBMMANrQNcy+B/yVlB/RzWLFjpHksrIKmyYxa36zRa0N7giqJdtufD+1Z5kBKdM3gI4ZQzOqX33ibz3zRSIi7MaKgk8hn8lLd1UC+k1dAXMyWjgGiiBOtoBboDLA9h8z9jzQtB6S0T3/0/lMnfFNhMGCAegES1SQBqSr6sAcz+rnRG5zQe20lghiJ9D2FURDr9tLXMzjp8+bahxoccIH35IFKWLciBt2Rt9c+BpWlF60hMxf+qeGyTgGwGlotvboyc3CWD67B9Gn8m3Y9SgsbpIw+tbNZ3OJEIyuRXzpDO0IaIBrdow9329sWkWy6RdXgW8AJFSDLerMo7O/ai9Rl8xNIZixO4yR/rDz1zs0Z13hEtutTeo9Bz/z9j7IJVlBArmRkW1LzGgd

In [ ]:
config2 = {'configurable':{'thread_id':'2'}}
workflow.invoke({'topic':'pasta'},config = config2)

NameError: name 'workflow' is not defined

In [ ]:
workflow.get_state(config2)

In [ ]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5", "checkpoint_ns": ""}}, {'topic':'samosa'})

In [ ]:
list(workflow.get_state_history(config1))

In [ ]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc72-ca16-6359-8001-7eea05e07dd2"}})

In [ ]:
list(workflow.get_state_history(config1))

In [11]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time



In [12]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [13]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [14]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)


In [ ]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)


In [1]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)


🔁 Re-running the graph to demonstrate fault tolerance...


NameError: name 'graph' is not defined

In [ ]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))